<a href="https://colab.research.google.com/github/be-anjosn/Estudos-IA/blob/main/Pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Usando a genAI do Google no Collabs





Quando programamos, deixar a chave da API escrita diretamente na string (`api_key="AIzaSy..."`) é um risco. Se você partilhar o ficheiro ou o enviar para o GitHub, qualquer pessoa pode roubar a sua quota.

A sua solução resolve isso usando **Variáveis de Ambiente** em conjunto com o **Cofre do Colab**:

- **O Cofre (`userdata`):** O Google Colab tem um gestor de segredos (o ícone da chave no menu lateral). O método `userdata.get()` vai lá buscar a chave de forma invisível.
- **A Variável de Ambiente (`os.environ`):** Em vez de passar a chave diretamente para a biblioteca, você "injeta" essa chave no sistema operativo temporário onde o seu código está a correr.
- **A Inteligência do SDK (`genai.Client()`):** Quando você chama o cliente vazio, a biblioteca da Google é programada para procurar automaticamente por uma variável de ambiente chamada `GOOGLE_API_KEY` (ou `GEMINI_API_KEY`). Como você a injetou na linha anterior, o cliente autentica-se magicamente.

Para adicionar uma chave do genAi, clique no ícone de chave ao lado
- Adicione um nome para sua chave
- Vá par ao site do [Google AI Studio - API keys](https://aistudio.google.com/api-keys)
- Faça sua autenticação no site. Clique em Criar Chave de API. Crie e copie o valor da chave para o valor no collabs
- Mude os valores no código para se conectar

In [7]:
import os # 1. Ferramenta para falar com o sistema operativo (Linux do Colab)
from google.colab import userdata # 2. Ferramenta para aceder ao cofre de senhas do Colab
from google import genai

# 3. O sistema operativo cria uma etiqueta chamada 'GOOGLE_API_KEY'
# 4. O valor dessa etiqueta é retirado em segurança da sua variável 'GEMINI_API_KEY_2'. Se necessário mude GEMINI_API_KEY_2 para o nome da sua chave
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY_2')

# 5. O cliente liga-se à Google. Ele olha para o sistema operativo,
# encontra a etiqueta 'GOOGLE_API_KEY', pega nela e usa-a para fazer login.
client = genai.Client()

#Para rodar uma query basica:
client.models.generate_content(model="gemini-flash-lite-latest", contents="O que é a Inteligência Artificial?")

#Também se pode criar um chatbot, que se insere varias querys e guarda num histórico
chatinho = client.chats.create(model="gemini-flash-lite-latest")
resposta = chatinho.send_message("Que dia é hoje?")

print("Resposta do chat:")
print(resposta.text)

print("-"*50)
print("\nHistórico do chat:")
chatinho.get_history()

Resposta do chat:
Como sou uma inteligência artificial, não tenho acesso ao dia atual em tempo real, pois isso depende do seu fuso horário. 

No entanto, você pode ver a data de hoje olhando no relógio ou calendário do seu dispositivo (computador ou celular)! Se precisar saber que dia da semana é, me diga a data (dia/mês/ano) e eu calculo para você.
--------------------------------------------------

Histórico do chat:


[UserContent(
   parts=[
     Part(
       text='Que dia é hoje?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text="""Como sou uma inteligência artificial, não tenho acesso ao dia atual em tempo real, pois isso depende do seu fuso horário. 
 
 No entanto, você pode ver a data de hoje olhando no relógio ou calendário do seu dispositivo (computador ou celular)! Se precisar saber que dia da semana é, me diga a data (dia/mês/ano) e eu calculo para você.""",
       thought_signature=b'\x12`\n^\x01i\x14}\x13\x8a\xdb\x9b\xf9\x83\xec\xab\xda\x8d\x8c\xd4\xcc\xaa4\x9c=&q^Z\xfcp\x94\xf6C\x82\xfa\x98,\xc9\x8e\x92L$\xcccDg\xf9\xb4o\xa6\xf2\xdfX!*\x03h:$\x04\xc4\x9c\xb1`{\xa1\xed\xcbSS\x84\t\x9b0l)\xd7\x83\xdc\x86\xbeD\\\xaa\xc8|\xd69\xad)z\xe7(Z\xc1 1'
     ),
   ],
   role='model'
 )]

# Fundamentos do Pydantic

O Pydantic é uma biblioteca do Python que faz **validação de dados** usando as anotações de tipo nativas da linguagem. Quando trabalhamos com LLMs (como o GPT, Gemini, Claude), o maior desafio é que eles geram texto livre. Se você precisa que o LLM extraia dados de um documento para salvar num banco de dados, você precisa de previsibilidade. É aí que o Pydantic entra: ele funciona como um "molde" estrito para a saída do LLM.

- **`BaseModel`:** É a classe principal da biblioteca. Quando você quer criar um formato de dados, você cria uma classe no Python que herda de `BaseModel`. Qualquer dado que entrar nessa classe será automaticamente validado e convertido (se possível) para o formato que você definiu.
- **Tipagem Nativa (`str`, `int`, `list`, etc.):** Você usa os tipos normais do Python para dizer ao molde o que você espera. Se você definir que `idade` é um `int` e o LLM tentar devolver a string `"30"`, o Pydantic é inteligente o suficiente para converter a string `"30"` para o número `30`. Se o LLM devolver `"trinta"`, o Pydantic bloqueia e gera um erro, protegendo seu sistema de dados inválidos.
- **Tipos Opcionais (`Optional[T]` ou `T | None`):** Textos do mundo real são imperfeitos. Se você pedir para o LLM extrair o "telefone" de um cliente de um e-mail, pode ser que o e-mail não tenha essa informação. Se você tipar apenas como `str`, o Pydantic vai dar erro se o dado faltar. Ao usar `str | None` (ou `Optional[str]`), você ensina ao Pydantic (e ao LLM, dependendo do framework usado) que esse campo pode ser nulo/inexistente, evitando que o seu código quebre.


Imagine que você está usando um LLM para ler resumos profissionais do LinkedIn e precisa extrair os dados em formato JSON para o seu sistema.

In [ ]:
from pydantic import BaseModel


# Criando o nosso "molde" com BaseModel
class PerfilProfissional(BaseModel):
    nome: str
    anos_experiencia: int
    habilidades: list[str]
    # Usando o | None (disponível no Python 3.10+) para indicar que é opcional.
    # É o mesmo que usar Optional[str] = None
    certificacao_principal: str | None = None

# Exemplo 1: O LLM encontrou todos os dados no texto
dados_do_llm = {
    "nome": "Ana Silva",
    "anos_experiencia": "5", # Note que veio como string, o Pydantic vai arrumar!
    "habilidades": ["Python", "Machine Learning"],
    "certificacao_principal": "AWS Certified"
}

perfil_1 = PerfilProfissional(**dados_do_llm)
print(perfil_1.anos_experiencia)
# Saída: 5 (convertido automaticamente para número inteiro)

# Exemplo 2: O LLM leu um texto que NÃO citava certificações
dados_incompletos = {
    "nome": "Carlos Souza",
    "anos_experiencia": 2,
    "habilidades": ["Excel", "Vendas"]
    # certificacao_principal está faltando
}

perfil_2 = PerfilProfissional(**dados_incompletos)
print(perfil_2.certificacao_principal)
# Saída: None (Não deu erro, pois avisamos que era opcional)

5
None


**Tarefa Prática:**

Cenário: Você está criando um assistente de IA que lê e-mails de reserva de hotéis e extrai as informações para o seu sistema de agendamento.

A sua tarefa: Escreva o código Python criando um `BaseModel` chamado `ReservaHotel`. Ele deve ser capaz de mapear as seguintes informações que o LLM vai extrair do e-mail:

1. O nome do hóspede.
2. A quantidade de noites que ele vai ficar.
3. Se a reserva inclui café da manhã ou não (Verdadeiro ou Falso).
4. Pedidos especiais (como "travesseiro extra" ou "quarto silencioso"). Como nem todo hóspede faz pedidos especiais, esse campo **pode não existir**.

In [ ]:
from pydantic import BaseModel

class ReservaHotel(BaseModel):
  nome: str
  quantidade_noites: int
  cafe_da_manha: bool
  pedidos_especiais: str | None = None


# Exemplo 1: Um e-mail detalhado com pedidos especiais
dados_reserva_1 = {
    "nome": "João Silva",
    "quantidade_noites": "3", # O Pydantic deve transformar isso em inteiro!
    "cafe_da_manha": "True",
    "pedidos_especiais": "Quarto em andar alto e longe do elevador"
}

# Exemplo 2: Um e-mail simples, sem nenhum pedido especial
dados_reserva_2 = {
    "nome": "Maria Oliveira",
    "quantidade_noites": 1,
    "cafe_da_manha": "False"
}

# Quando você criar a classe ReservaHotel, seu teste será assim:
reserva1 = ReservaHotel(**dados_reserva_1)
reserva2 = ReservaHotel(**dados_reserva_2)

print(reserva1)
print(reserva2)


nome='João Silva' quantidade_noites=3 cafe_da_manha=True pedidos_especiais='Quarto em andar alto e longe do elevador'
nome='Maria Oliveira' quantidade_noites=1 cafe_da_manha=False pedidos_especiais=None


# Direcionamento do LLM (Engenharia de Prompt via Tipagem)

O Pydantic permite transformar a tipagem de dados em **engenharia de prompt embutida**. Quando conectamos o Pydantic a um LLM, o modelo de inteligência artificial lê o esquema JSON gerado pela classe antes de formular a resposta. Usar atributos avançados instrui o modelo sobre *como* extrair e classificar os dados, reduzindo alucinações e garantindo formatos rígidos.

- **`Field(description="...")`:** Funciona como um micro-prompt exclusivo para um único campo. Em vez de escrever instruções longas no prompt principal (ex: "extraia o nome da empresa, mas ignore filiais"), você embute essa regra diretamente na variável. O LLM lê a descrição e ajusta a sua extração para aquele campo específico.
- **Restrições no `Field` (`ge/le`, `min_length/max_length`):** Impõem limites físicos à saída do LLM para evitar respostas fora do escopo.
    - `ge` (greater or equal / maior ou igual) e `le` (less or equal / menor ou igual) limitam intervalos numéricos.
    - `min_length` e `max_length` controlam o tamanho de strings.
    - Se o LLM tentar alucinar uma nota "11" num sistema de 1 a 10, o Pydantic bloqueia e (em frameworks como LangChain ou LlamaIndex) devolve o erro ao LLM para que ele se corrija automaticamente.
- **`Literal` e `Enum`:** São as ferramentas definitivas para classificação categórica. Ao definir um campo como `Literal["urgente", "normal"]`, você retira a liberdade criativa do LLM. Ele não poderá responder "alta prioridade" ou "crítico"; ele é forçado estruturalmente a escolher exatamente uma das opções fornecidas, facilitando o roteamento no seu sistema.

Imagine um sistema que lê avaliações de clientes e precisa categorizar o sentimento, dar uma nota matemática e resumir o texto para o painel de métricas.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AnaliseAvaliacao(BaseModel):
    # Limitando as opções a três valores exatos e rígidos
    sentimento: Literal["positivo", "neutro", "negativo"] = Field(description="O sentimento geral do cliente em relação ao serviço.")

    # Forçando o LLM a dar uma nota matemática contida entre 1 e 5
    nota_atendimento: int = Field(
        ge=1,
        le=5,
        description="Nota de 1 a 5 baseada no quão satisfeito o cliente parece estar."
    )

    # Controlando o tamanho do texto gerado e dando uma diretriz clara
    resumo_curto: str = Field(
        max_length=100,
        description="Resuma o problema ou elogio do cliente em no máximo 100 caracteres. Vá direto ao ponto."
    )

Neste cenário, se um cliente escreve um texto imenso e confuso, o seu sistema não quebra. O LLM é obrigado a devolver um sentimento exato para alimentar os seus gráficos, uma nota entre 1 e 5 e um resumo cortado no tamanho certo para caber na interface do seu software.

**Tarefa Prática:**

Cenário: Você está a construir um classificador automático para tíquetes de suporte técnico de TI (Helpdesk). Quando um funcionário relata um problema (ex: *"Meu computador não liga e a tela está piscando"*), o LLM precisa de classificar a mensagem para acionar a equipa correta.

A sua tarefa: Crie uma classe Pydantic chamada `TicketSuporte` utilizando `Field` e `Literal` para estruturar a extração dos seguintes dados:

1. **`departamento`**: Deve obrigatoriamente ser um dos seguintes: `"Hardware"`, `"Software"` ou `"Redes"`.
2. **`urgencia`**: Um número inteiro de 1 a 10 (onde 1 é muito baixo e 10 é crítico). Use as restrições corretas no `Field`.
3. **`tags_problema`**: Uma lista de strings (`list[str]`). Adicione uma `description` no `Field` instruindo o LLM a extrair, no máximo, 3 palavras-chave curtas baseadas no problema relatado.

In [8]:
from pydantic import BaseModel, Field
from typing import Literal
from google import genai
from google.genai import types



# 1. A sua classe Pydantic (O Molde)
class TicketSuporte(BaseModel):
    departamento: Literal["Hardware", "Software", "Rede"] = Field(
        description="Qual o Departamento que pertence aquele ticket"
    )
    urgencia: int = Field(
        ge=1, le=10,
        description="Um número inteiro de 1 a 10, que classifica a urgência do problema"
    )
    tags_problema: list[str] = Field(
        description="Extraia no máximo, 3 palavras-chave curtas baseadas no problema relatado."
    )

# 2. Inicializar o cliente Gemini
# Substitua pela sua chave real ou defina a variável de ambiente GEMINI_API_KEY
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY_2')
client = genai.Client()

# 3. O problema real relatado por um utilizador
mensagem_utilizador = """
Malta da TI, socorro! Estava a usar o Excel e de repente o ecrã ficou todo azul
e o computador desligou-se com cheiro a queimado. Tenho uma apresentação
daqui a 10 minutos com o diretor, preciso de outro portátil imediatamente!
"""

# 4. A chamada à IA
resposta = client.models.generate_content(
    model='gemini-flash-lite-latest',
    contents=f"Analise este pedido de suporte e extraia os dados: {mensagem_utilizador}",
    config=types.GenerateContentConfig(
        # Forçamos a saída a ser em JSON
        response_mime_type="application/json",
        # Passamos a sua classe Pydantic para a IA saber exatamente o que queremos
        response_schema=TicketSuporte,
        # Temperatura baixa (0.1) evita que a IA seja "criativa" e foque apenas na extração
        temperature=0.1,
    ),
)

# 5. O Gemini devolve um texto em formato JSON.
# Usamos o model_validate_json para transformar esse texto de volta no seu objeto Python validado!
ticket_extraido = TicketSuporte.model_validate_json(resposta.text)

print("--- Resultado da Extração da IA ---")
print(f"Departamento: {ticket_extraido.departamento}")
print(f"Urgência: {ticket_extraido.urgencia}")
print(f"Tags: {ticket_extraido.tags_problema}")

--- Resultado da Extração da IA ---
Departamento: Hardware
Urgência: 10
Tags: ['computador', 'queimado', 'desligou']


# Estruturas de Dados Complexas

Quando pede uma pizza, a informação não é apenas uma linha de texto. Ela tem partes dentro de partes. É exatamente isso que o Pydantic resolve aqui.

- **Modelos Aninhados (Nested Models):** Imagine que você tem uma caixa grande (o seu molde principal). Em vez de jogar as coisas lá dentro de forma desorganizada, você coloca caixas menores lá dentro.
    - *No código:* Você cria um molde pequeno chamado `Endereco` (com rua, número, cidade). Depois, cria um molde maior chamado `Pedido` e diz: "Dentro do Pedido, vai haver uma variável que é do tipo `Endereco`".
- **Extração de Múltiplos Itens (`list[Molde]`):** E se o cliente pedir 3 pizzas e 2 refrigerantes? Você não sabe de antemão quantos itens ele vai pedir.
    - *No código:* Você cria um molde pequeno chamado `Produto`. No molde principal, você diz que quer uma **lista** de Produtos (`list[Produto]`). O LLM vai ler o texto e criar quantos Produtos forem necessários, guardando tudo nessa lista.
- **Union (O poder de escolha):** A palavra *Union* significa união, mas na prática funciona como um **"OU"**. Serve para quando o cenário é incerto.
    - *No código:* Você diz ao LLM: "Se o pedido estiver certo, preencha o molde `PedidoValido`. **OU** (`|`), se o texto for uma reclamação ou brincadeira, preencha o molde `PedidoRecusado`". O LLM lê o texto e **decide** qual dos dois moldes usar.

Vamos ver como as "caixas menores" entram na "caixa maior", e como damos ao LLM o poder de escolher o molde correto.

In [ ]:
from pydantic import BaseModel
from typing import Literal

# --- CAIXAS MENORES (Modelos Aninhados) ---

# Molde 1: O endereço (Caixa pequena 1)
class Endereco(BaseModel):
    rua: str
    numero: str

# Molde 2: Um item do pedido (Caixa pequena 2)
class Item(BaseModel):
    nome_da_comida: str
    quantidade: int

# --- AS DUAS OPÇÕES DO NOSSO "UNION" (OU) ---

# Opção A: O texto é um pedido real
class PedidoValido(BaseModel):
    # A etiqueta (Literal) que avisa que este é o molde de sucesso
    status: Literal["sucesso"]

    # Colocamos a caixa pequena (Endereco) dentro da caixa grande
    local_de_entrega: Endereco

    # Dizemos que queremos uma LISTA de itens, não apenas um
    comidas: list[Item]

# Opção B: O texto é uma pessoa a perguntar "A que horas fecham?"
class PedidoInvalido(BaseModel):
    # A etiqueta que avisa que o LLM escolheu o molde de erro
    status: Literal["erro"]
    motivo: str

# --- A UNIÃO FINAL ---
# O LLM vai olhar para o texto e decidir: Preencho o Válido OU o Inválido?
MoldeParaO_LLM = PedidoValido | PedidoInvalido

Teste:


In [ ]:
import os
from google.colab import userdata
from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Literal

# 1. Autenticação no Colab
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY_2')
cliente = genai.Client()

# ==========================================
# 2. DEFINIÇÃO DAS CAIXAS (MOLDES)
# ==========================================

class Endereco(BaseModel):
    rua: str
    numero: str

class Item(BaseModel):
    nome_da_comida: str
    quantidade: int

class PedidoValido(BaseModel):
    status: Literal["sucesso"]
    local_de_entrega: Endereco
    comidas: list[Item]

class PedidoInvalido(BaseModel):
    status: Literal["erro"]
    motivo: str


# A Caixa Principal que envolve a decisão (Obrigatório para o Gemini)
class ProcessamentoPedido(BaseModel):
    resultado: PedidoValido | PedidoInvalido

# ==========================================
# 3. FUNÇÃO DE TESTE E ROTEAMENTO
# ==========================================

def analisar_mensagem(texto_do_cliente):
    print(f"\n💬 Mensagem recebida: '{texto_do_cliente}'")
    print("A pensar...")

    resposta = cliente.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=f"Analise o pedido. Se for uma compra válida, extraia os dados. Se for uma pergunta ou pedido sem sentido, rejeite e explique o motivo. Mensagem: {texto_do_cliente}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ProcessamentoPedido,
            temperature=0.1,
        ),
    )

    # Convertendo a resposta JSON para o nosso objeto Python
    dados = ProcessamentoPedido.model_validate_json(resposta.text)

    # -----------------------------------------------------
    # ROTEAMENTO: O sistema lê a etiqueta que o LLM escolheu
    # -----------------------------------------------------
    if dados.resultado.status == "sucesso":
        print("✅ O Gemini usou o molde 'PedidoValido'!")
        print(f"📍 Entrega em: {dados.resultado.local_de_entrega.rua}, {dados.resultado.local_de_entrega.numero}")
        for item in dados.resultado.comidas:
            print(f"🍕 Preparar: {item.quantidade}x {item.nome_da_comida}")

    elif dados.resultado.status == "erro":
        print("❌ O Gemini usou o molde 'PedidoInvalido'!")
        print(f"🤖 Resposta da IA: {dados.resultado.motivo}")
    print("-" * 50)

# ==========================================
# 4. EXECUTANDO OS TESTES
# ==========================================

# Cenário 1: Um pedido real e válido
analisar_mensagem("Quero 2 pizzas marguerita para a Rua das Flores, 123")

# Cenário 2: Uma pergunta aleatória (o LLM vai rejeitar)
analisar_mensagem("Vendem hambúrguer?")


💬 Mensagem recebida: 'Quero 2 pizzas marguerita para a Rua das Flores, 123'
A pensar...
✅ O Gemini usou o molde 'PedidoValido'!
📍 Entrega em: Rua das Flores, 123
🍕 Preparar: 2x pizza marguerita
--------------------------------------------------

💬 Mensagem recebida: 'Vendem hambúrguer?'
A pensar...
❌ O Gemini usou o molde 'PedidoInvalido'!
🤖 Resposta da IA: O pedido e apenas uma pergunta sobre o cardapio e nao uma solicitacao de compra.
--------------------------------------------------


**Teste Prático**

Vamos treinar **apenas os Modelos Aninhados e a Lista** para fixar bem a ideia das "caixas dentro de caixas". Deixaremos o Union de lado neste exercício.

Cenário: Você está a criar um sistema para ler resumos de livros.

A sua tarefa: Escreva o código Python com dois moldes (classes):

1. Crie a caixa pequena: Faça uma classe chamada `Autor` que tenha `nome` (str) e `ano_nascimento` (int).
2. Crie a caixa grande: Faça uma classe chamada `Livro`. Ela deve ter:
    - `titulo` (str)
    - `escritor` (Esta variável deve usar o modelo **`Autor`** que você criou acima).
    - `capitulos` (Esta variável deve ser uma lista de strings, ou seja, **`list[str]`**).

Tente escrever apenas a estrutura dessas duas classes e envie aqui. Se entender como colocar o `Autor` dentro do `Livro`, você dominou a parte mais importante das estruturas complexas!

In [ ]:
import os
from google.colab import userdata
from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Literal

# 1. Autenticação no Colab
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY_2')
cliente = genai.Client()


# --- CAIXAS MENORES (Modelos Aninhados) ---

# Molde 1: O Autor (Caixa pequena 1)
class Autor(BaseModel):
    nome: str
    ano_nascimento: str


# --- AS DUAS OPÇÕES DO NOSSO "UNION" (OU) ---

# Opção A: O texto é um pedido real
class LivroValido (BaseModel):
    # A etiqueta (Literal) que avisa que este é o molde de sucesso
    status: Literal["sucesso"]
    titulo: str
    escritor: Autor
    capitulos: list[str]

class LivroInvalido(BaseModel):
    status: Literal["erro"]
    motivo: str

# A Caixa Principal que envolve a decisão (Obrigatório para o Gemini)
class Processamento(BaseModel):
    resultado: LivroValido | LivroInvalido



# ==========================================
# 3. FUNÇÃO DE TESTE E ROTEAMENTO
# ==========================================

def analisar_mensagem(texto_do_cliente):
    print(f"\n💬 Mensagem recebida: '{texto_do_cliente}'")
    print("A pensar...")

    resposta = cliente.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=f"Analise o Livro. Se o livro existir, extraia os dados. Se o livro não existir rejeite e explique o motivo. Mensagem: {texto_do_cliente}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Processamento,
            temperature=0.1,
        ),
    )
    print(resposta.text)

    # Convertendo a resposta JSON para o nosso objeto Python
    dados = Processamento.model_validate_json(resposta.text)

    # -----------------------------------------------------
    # ROTEAMENTO: O sistema lê a etiqueta que o LLM escolheu
    # -----------------------------------------------------
    if dados.resultado.status == "sucesso":
        print("✅ O Gemini usou o molde 'LivroValido'!")
        print(f"AUTOR: {dados.resultado.escritor.nome}")
        print(f"LIVRO: {dados.resultado.titulo}")
        for item in dados.resultado.capitulos:
            print(f"Capitulos: {item}")

    elif dados.resultado.status == "erro":
        print("❌ O Gemini usou o molde 'LivroInvalido'!")
        print(f"🤖 Resposta da IA: {dados.resultado.motivo}")
    print("-" * 50)

# ==========================================
# 4. EXECUTANDO OS TESTES
# ==========================================

# Cenário 1: Um pedido real e válido
analisar_mensagem("Livro: Orgulho e Preconceito")

# Cenário 2: Uma pergunta aleatória (o LLM vai rejeitar)
analisar_mensagem("Uma noite no chiqueiro")



💬 Mensagem recebida: 'Livro: Orgulho e Preconceito'
A pensar...
{
  "resultado": {
    "status": "sucesso",
    "titulo": "Orgulho e Preconceito",
    "escritor": {
      "nome": "Jane Austen",
      "ano_nascimento": "1775"
    },
    "capitulos": [
      "Capitulo 1",
      "Capitulo 2",
      "Capitulo 3"
    ]
  }
}
✅ O Gemini usou o molde 'LivroValido'!
AUTOR: Jane Austen
LIVRO: Orgulho e Preconceito
Capitulos: Capitulo 1
Capitulos: Capitulo 2
Capitulos: Capitulo 3
--------------------------------------------------

💬 Mensagem recebida: 'Uma noite no chiqueiro'
A pensar...
{
  "resultado": {
    "status": "erro",
    "motivo": "O livro 'Uma noite no chiqueiro' não foi encontrado na base de dados."
  }
}
❌ O Gemini usou o molde 'LivroInvalido'!
🤖 Resposta da IA: O livro 'Uma noite no chiqueiro' não foi encontrado na base de dados.
--------------------------------------------------


# Validação Customizada e Correção



Até agora, vimos como o Pydantic garante que a saída do LLM tenha o *formato* correto (texto, número, lista). No entanto, às vezes o dado tem o tipo certo, mas não faz sentido para a regra de negócio da sua empresa. Os validadores servem para intervir **depois** que o LLM gera a resposta, mas **antes** de o dado entrar no seu sistema.

- **`@field_validator` (Regra de um campo só):** É uma função que atua como um "filtro de limpeza" para uma variável específica. Se o LLM extrair um número de telefone com espaços ou traços, ou escrever um nome todo em minúsculas, você usa este validador para formatar o texto, remover acentos ou padronizar a informação (ex: forçar tudo a letras maiúsculas).
- **`@model_validator` (Regra de múltiplos campos):** É usado quando a validação de um campo **depende de outro**. Imagine que você está extraindo informações de um contrato de trabalho. Se a variável `tipo_contrato` for `"Estágio"`, a variável `carga_horaria_semanal` não pode ser superior a 30 horas. O validador de modelo cruza essas duas informações e bloqueia a operação se a regra for quebrada.
- **`ValidationError` (O Ciclo de Autocorreção):** Quando um LLM alucina e quebra uma das suas regras, o Pydantic gera um alerta chamado `ValidationError`. Em sistemas de IA profissionais, nós usamos o bloco `try... except` para capturar essa mensagem exata de erro e enviá-la de volta ao LLM de forma automática (ex: *"A sua extração falhou com este erro: 'Estagiários não podem trabalhar mais de 30h'. Por favor, corrija os dados."*). Isso cria um "Self-Correction Loop" onde a IA se corrige sozinha sem intervenção humana.

Vamos simular um sistema que extrai regras de cupons de desconto criados por uma equipe de marketing. O código de cupom precisa estar sempre em maiúsculas e sem espaços, e descontos gigantescos exigem autorização.

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError

class CupomDesconto(BaseModel):
    codigo: str
    porcentagem_desconto: int = Field(ge=1, le=100)

    # 1. VALIDADOR DE CAMPO (@field_validator)
    # Roda apenas na variável 'codigo' para limpá-la
    @field_validator('codigo')
    @classmethod
    def limpar_codigo(cls, valor_extraido):
        # Remove espaços nas pontas, troca espaços no meio por "_" e põe tudo em maiúsculas
        codigo_limpo = valor_extraido.strip().replace(" ", "_").upper()
        return codigo_limpo

    # 2. VALIDADOR DE MODELO (@model_validator)
    # Roda depois de tudo. O 'self' contém o modelo inteiro preenchido.
    @model_validator(mode='after')
    def verificar_regra_desconto(self):
        # Regra de negócio: Se o desconto for maior que 50%, o código DEVE começar com "SUPER"
        if self.porcentagem_desconto > 50 and not self.codigo.startswith("SUPER"):
            # Ao levantar um ValueError, o Pydantic transforma isso num ValidationError
            raise ValueError("Descontos acima de 50% devem ter o código iniciando com 'SUPER'.")

        return self # Retorna o modelo validado se tudo estiver certo

# --- TESTES E CAPTURA DE ERRO (Self-Correction base) ---

# Cenário A: O LLM extraiu os dados com formatação suja
dados_llm_1 = {
    "codigo": "  verao 2026 ",
    "porcentagem_desconto": 20
}

cupom1 = CupomDesconto(**dados_llm_1)
print(f"✅ Cupom 1 validado: O Pydantic corrigiu o código para '{cupom1.codigo}'")

# Cenário B: O LLM alucinou um desconto de 80% num cupom normal
dados_llm_2 = {
    "codigo": "fim de ano",
    "porcentagem_desconto": 80
}

try:
    cupom2 = CupomDesconto(**dados_llm_2)
except ValidationError as e:
    print("\n❌ O Pydantic bloqueou a IA! Capturamos o seguinte erro:")
    print(e)
    # Num sistema real, enviaríamos este erro 'e' de volta ao Gemini
    # num novo prompt pedindo para ele ajustar a extração.

✅ Cupom 1 validado: O Pydantic corrigiu o código para 'VERAO_2026'

❌ O Pydantic bloqueou a IA! Capturamos o seguinte erro:
1 validation error for CupomDesconto
  Value error, Descontos acima de 50% devem ter o código iniciando com 'SUPER'. [type=value_error, input_value={'codigo': 'fim de ano', ...rcentagem_desconto': 80}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


**Teste Prático:**

Cenário: Você está a automatizar a extração de dados para um sistema de agendamento de consultas médicas.

A sua tarefa: Crie uma classe chamada `ConsultaMedica` que recebe três dados extraídos pelo LLM:

1. `nome_paciente` (str)
2. `idade` (int)
3. `especialidade` usando `Literal["pediatria", "cardiologia"]`.

**As regras que você deve programar:**

- Crie um `@field_validator` para o `nome_paciente`. A regra é simples: o nome deve ser convertido para ter sempre as primeiras letras maiúsculas (Dica: em Python, a função de string `.title()` faz isso. Exemplo: `"joão silva".title()` vira `"João Silva"`).
- Crie um `@model_validator(mode='after')`. A regra de negócio é: se a `especialidade` for `"pediatria"`, a `idade` **tem de ser** estritamente menor que 18 anos. Se for 18 ou mais, levante um `ValueError` com a mensagem: *"Pacientes com 18 anos ou mais não podem ser atendidos na pediatria."*

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError

class ConsultaMedica(BaseModel):
    nome_paciente: str
    idade: int
    especialidade: Literal["pediatria", "cardiologia"]

    # 1. VALIDADOR DE CAMPO (@field_validator)
    # Roda apenas na variável 'codigo' para limpá-la
    @field_validator('nome_paciente')
    @classmethod
    def limpar_codigo(cls, valor_extraido):
        nome_paciente = valor_extraido.title()
        return nome_paciente

    # 2. VALIDADOR DE MODELO (@model_validator)
    # Roda depois de tudo. O 'self' contém o modelo inteiro preenchido.
    @model_validator(mode='after')
    def verificar_regra_desconto(self):
        # Regra de negócio: Se a especialidade for "pediatria", a idade tem de ser estritamente menor que 18 anos.
        if self.especialidade == "pediatria" and self.idade >= 18:
            # Ao levantar um ValueError, o Pydantic transforma isso num ValidationError
            raise ValueError("Pacientes com 18 anos ou mais não podem ser atendidos na pediatria")

        return self # Retorna o modelo validado se tudo estiver certo

# --- TESTES E CAPTURA DE ERRO (Self-Correction base) ---

# Cenário A: O LLM extraiu os dados com formatação suja
dados_llm_1 = {
    "nome_paciente": "junior lima",
    "idade": 14,
    "especialidade": "pediatria"
}

consulta1 = ConsultaMedica(**dados_llm_1)
print(f"✅ Cupom 1 validado: O Pydantic corrigiu o código para '{consulta1.nome_paciente}'")



# Cenário B: O LLM alucinou um desconto de 80% num cupom normal
dados_llm_2 = {
    "nome_paciente": "junior lima",
    "idade": 19,
    "especialidade": "pediatria"
}

try:
    consulta2 = ConsultaMedica(**dados_llm_2)
except ValidationError as e:
    print("\n❌ O Pydantic bloqueou a IA! Capturamos o seguinte erro:")
    print(e)
    # Num sistema real, enviaríamos este erro 'e' de volta ao Gemini
    # num novo prompt pedindo para ele ajustar a extração.

✅ Cupom 1 validado: O Pydantic corrigiu o código para 'Junior Lima'

❌ O Pydantic bloqueou a IA! Capturamos o seguinte erro:
1 validation error for ConsultaMedica
  Value error, Pacientes com 18 anos ou mais não podem ser atendidos na pediatria [type=value_error, input_value={'nome_paciente': 'junior...cialidade': 'pediatria'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


# Serialização e Parseamento (A interface com a API)

Pense no Pydantic como um **tradutor universal** que fica na porta do seu sistema. Os Modelos de Linguagem (LLMs) comunicam através de texto (strings JSON). No entanto, o seu código em Python, o seu banco de dados e o seu site (frontend) não lidam bem com texto solto; eles precisam de estruturas organizadas. É aqui que entram os métodos de serialização.

- **`model_json_schema()` (O Manual de Instruções para a IA):** Os LLMs não conseguem ler código Python. Quando usamos a OpenAI, Anthropic, ou configuramos ferramentas customizadas, precisamos de traduzir o nosso "molde Pydantic" para um formato universal chamado *JSON Schema*. Este método faz isso automaticamente. Ele gera um dicionário detalhado explicando quais campos existem, se são obrigatórios e quais os limites numéricos, para que o LLM saiba exatamente o que gerar. *(Nota: O SDK do Gemini já chama este método por debaixo dos panos quando você passa a classe no `response_schema`).*
- **`model_validate_json()` (A Porta de Entrada):** A IA terminou de pensar e devolveu-lhe um bloco de texto que se parece com JSON. Você nunca deve confiar cegamente nesse texto. Ao passar essa string por este método, o Pydantic vai convertê-la em JSON, verificar todas as regras e, se passar, transformá-la num **objeto Python seguro** (onde você pode fazer `objeto.nome`). Se houver erro, ele levanta o `ValidationError`.
- **`model_dump()` e `model_dump_json()` (A Porta de Saída):** Depois de receber e validar o dado da IA, você precisa de o guardar ou enviar para algum lado.
    - `model_dump()`: Transforma o seu objeto seguro num **dicionário Python** (`dict`). É o formato perfeito para guardar num banco de dados como MongoDB ou PostgreSQL (via SQLAlchemy).
    - `model_dump_json()`: Transforma o seu objeto seguro numa **string de texto JSON**. É o formato perfeito para enviar esse dado via API para o site (frontend) mostrar ao utilizador.

Vamos simular o ciclo completo de um dado: gerar o esquema para a IA, receber a string da IA, validar para Python e, por fim, exportar para o banco de dados.

In [ ]:
from pydantic import BaseModel, Field
import json

class AvaliacaoProduto(BaseModel):
    produto_id: int
    sentimento: str = Field(pattern="^(Positivo|Negativo|Neutro)$") # Regex: aceita apenas estas 3 palavras
    nota: float
    comentario: str

# 1. GERANDO O ESQUEMA PARA A API DA IA (O que a IA vai ler)
# É isto que frameworks como LangChain ou APIs da OpenAI usam nos bastidores.
esquema_para_llm = AvaliacaoProduto.model_json_schema()
print("--- 1. O Esquema enviado para a IA ---")
# Usamos json.dumps apenas para imprimir bonito no ecrã
print(json.dumps(esquema_para_llm, indent=2, ensure_ascii=False))

# ---------------------------------------------------------
# 2. A PORTA DE ENTRADA (Parseamento)
# Simulando a string de texto crua que o LLM (Gemini/ChatGPT) nos devolveu
texto_gerado_pela_ia = '{"produto_id": 994, "sentimento": "Positivo", "nota": 4.5, "comentario": "Muito bom, mas a bateria dura pouco."}'

print("\n--- 2. Validando a resposta da IA ---")
# Convertendo a string solta num Objeto Python validado e seguro
avaliacao_segura = AvaliacaoProduto.model_validate_json(texto_gerado_pela_ia)
print(f"Objeto Python gerado com sucesso! Nota extraída: {avaliacao_segura.nota}")

# ---------------------------------------------------------
# 3. A PORTA DE SAÍDA (Serialização)
# Agora queremos salvar isto no nosso banco de dados MongoDB
print("\n--- 3. Exportando para o Banco de Dados ---")
# model_dump() converte a classe de volta para um Dicionário limpo
dicionario_para_banco = avaliacao_segura.model_dump()

print(type(dicionario_para_banco)) # Vai mostrar <class 'dict'>
print(dicionario_para_banco)

--- 1. O Esquema enviado para a IA ---
{
  "properties": {
    "produto_id": {
      "title": "Produto Id",
      "type": "integer"
    },
    "sentimento": {
      "pattern": "^(Positivo|Negativo|Neutro)$",
      "title": "Sentimento",
      "type": "string"
    },
    "nota": {
      "title": "Nota",
      "type": "number"
    },
    "comentario": {
      "title": "Comentario",
      "type": "string"
    }
  },
  "required": [
    "produto_id",
    "sentimento",
    "nota",
    "comentario"
  ],
  "title": "AvaliacaoProduto",
  "type": "object"
}

--- 2. Validando a resposta da IA ---
Objeto Python gerado com sucesso! Nota extraída: 4.5

--- 3. Exportando para o Banco de Dados ---
<class 'dict'>
{'produto_id': 994, 'sentimento': 'Positivo', 'nota': 4.5, 'comentario': 'Muito bom, mas a bateria dura pouco.'}


#Aplicação Direta no Ecossistema de IA

Hoje em dia, a IA não serve apenas para conversar; ela serve para **agir**. E para agir num sistema real, ela precisa de regras estritas.

- **Structured Outputs (Saídas Estruturadas):** Antigamente, os programadores escreviam prompts como: *"Por favor, responda apenas em JSON, não adicione nenhum texto antes ou depois"*. Quase sempre a IA falhava. Hoje, graças ao Pydantic, usamos **Structured Outputs**. Como viu nos exemplos do Gemini (`response_schema`), nós entregamos a classe Pydantic diretamente à API. A API usa o esquema para "bloquear" as palavras do LLM a nível de processador, forçando-o a gerar um JSON 100% perfeito, sempre.
- **Tool / Function Calling (Uso de Ferramentas):** Imagine que quer que a IA envie um e-mail. A IA não consegue enviar o e-mail sozinha, ela precisa de acionar uma função Python que você escreveu. O problema: como garantir que a IA lhe passa um destinatário válido e não apenas texto livre? A solução: **Modelamos os argumentos da função como uma classe Pydantic**. O LLM preenche a classe, o Pydantic valida, e o seu código Python executa a ação com segurança.
- **Integrações Chave (`Instructor` e `Pydantic AI`):** O Pydantic tornou-se um padrão tão absoluto na inteligência artificial que bibliotecas inteiras nasceram em torno dele:
    - **Instructor:** Uma biblioteca famosíssima que simplifica a extração. Ela "remenda" as APIs (da OpenAI, Anthropic, Gemini) para que, em vez de você fazer o parseamento manual, a chamada à API já devolva o objeto Pydantic pronto e validado numa única linha de código.
    - **Pydantic AI:** O mais recente framework construído pela própria equipa criadora do Pydantic, feito especificamente para criar Agentes autônomos onde o roteamento, a memória e as ferramentas são todos controlados por moldes Pydantic.

Vamos simular como prepararíamos a configuração de uma ferramenta para a IA usar. O cenário é um assistente que agenda reuniões no calendário.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# 1. A FUNÇÃO REAL (O que o seu servidor vai executar)
def criar_evento_calendario(titulo: str, dia: str, duracao_minutos: int):
    # Aqui entraria o código real para ligar à API do Google Calendar, por exemplo
    print(f"📅 [AÇÃO] Marcado: '{titulo}' no dia {dia} por {duracao_minutos} minutos.")
    return "Sucesso"

# 2. O MOLDE PYDANTIC (A "interface" da ferramenta para a IA ler)
# Criamos uma classe que representa exatamente os argumentos da função acima.
class FerramentaAgendamento(BaseModel):
    titulo: str = Field(description="O assunto curto da reunião.")
    dia: str = Field(description="Data no formato YYYY-MM-DD.")
    duracao_minutos: Literal[15, 30, 45, 60] = Field(
        description="A duração permitida para reuniões."
    )

# 3. O FLUXO (Simulação)
# Se o utilizador disser: "Marca aí um papo rápido de 30 min sobre Vendas amanhã"
# A IA geraria este JSON baseado na nossa classe FerramentaAgendamento:
json_gerado_pela_ia = '{"titulo": "Alinhamento de Vendas", "dia": "2026-10-09", "duracao_minutos": 30}'

# O Pydantic valida os dados que a IA gerou e transforma-os num objeto Python
argumentos_seguros = FerramentaAgendamento.model_validate_json(json_gerado_pela_ia)

# Finalmente, desempacotamos (**) o dicionário seguro diretamente para a função real!
# Se a IA tivesse alucinado uma reunião de "90" minutos, o Pydantic teria bloqueado
# antes de chegar a esta linha, protegendo a nossa função e o nosso calendário.
criar_evento_calendario(**argumentos_seguros.model_dump())